### Flight Risk - Data Preparation

In [1]:
import os

import pandas as pd
import numpy as np

from pathlib import Path

from dotenv import load_dotenv

In [2]:
# ENV vars
load_dotenv()

START_YEAR = os.environ.get('START_YEAR')
END_YEAR = os.environ.get('END_YEAR')

In [3]:
# DATA PATH

data_path = Path.cwd().parent / ".data" / f"vra_{START_YEAR}_to_{END_YEAR}.csv"
data_path

PosixPath('/home/beck/code/LuisHBeck/flight-risk/.data/vra_2022_to_2025.csv')

In [4]:
# Columns Mapping

columns_mapping = {
    "ICAO Empresa Aérea": "airline_icao",
    "Código Tipo Linha": "flight_type_code",
    "ICAO Aeródromo Origem": "origin_icao",
    "ICAO Aeródromo Destino": "destination_icao",
    "Partida Prevista": "dep_scheduled",
    "Partida Real": "dep_actual",
    "Chegada Prevista": "arr_scheduled",
    "Chegada Real": "arr_actual",
    "Situação Voo": "flight_status"
}

datetime_cols = [
    "dep_scheduled",
    "dep_actual",
    "arr_scheduled",
    "arr_actual"
]

In [5]:
# Loading raw data

data = (
    pd.read_csv(
        data_path,
        sep=";",
        usecols=columns_mapping.keys(),
        low_memory=False
    )
    .rename(columns=columns_mapping)
)

data.head(5)

,airline_icao,flight_type_code,origin_icao,destination_icao,dep_scheduled,dep_actual,arr_scheduled,arr_actual,flight_status
0,TAM,N,SBRJ,SBGR,2022-01-06 14:20:00,2022-01-06 14:31:00,2022-01-06 15:25:00,2022-01-06 15:29:00,REALIZADO
1,TAM,N,SBRJ,SBGR,2022-01-07 14:20:00,2022-01-07 14:47:00,2022-01-07 15:25:00,2022-01-07 15:42:00,REALIZADO
2,TAM,N,SBRJ,SBGR,2022-01-08 14:20:00,2022-01-08 14:12:00,2022-01-08 15:25:00,2022-01-08 15:12:00,REALIZADO
3,TAM,N,SBRJ,SBGR,2022-01-09 14:20:00,2022-01-09 14:16:00,2022-01-09 15:25:00,2022-01-09 15:15:00,REALIZADO
4,TAM,N,SBRJ,SBGR,2022-01-11 14:20:00,2022-01-11 14:20:00,2022-01-11 15:25:00,2022-01-11 15:15:00,REALIZADO


In [6]:
data.shape

(3851504, 9)

In [7]:
# Filters

# Only the flights that were carried out
data = data[data["flight_status"] == "REALIZADO"]

# Only national domestic flights (N)
data = data[data["flight_type_code"] == "N"]

data = data.drop(columns=["flight_status", "flight_type_code"])

In [8]:
data.shape

(3057454, 7)

In [9]:
# Removing trash data

# Duplicates
data = data.drop_duplicates()

# Invalid values
data = data.dropna(
    subset=[
        "dep_scheduled",
        "dep_actual",
        "arr_scheduled",
        "arr_actual"
    ]
)

In [10]:
data.shape

(2984123, 7)

In [11]:
# Convert DateTime columns
data[datetime_cols] = (
    data[datetime_cols].apply(
        lambda col: pd.to_datetime(col, format="ISO8601") \
        .dt.floor("s") \
        .astype("datetime64[ns]")
    )
)

In [12]:
data.dtypes

airline_icao                   str
origin_icao                    str
destination_icao               str
dep_scheduled       datetime64[ns]
dep_actual          datetime64[ns]
arr_scheduled       datetime64[ns]
arr_actual          datetime64[ns]
dtype: object

In [13]:
data.shape

(2984123, 7)

In [14]:
data.head()

,airline_icao,origin_icao,destination_icao,dep_scheduled,dep_actual,arr_scheduled,arr_actual
0,TAM,SBRJ,SBGR,2022-01-06 14:20:00,2022-01-06 14:31:00,2022-01-06 15:25:00,2022-01-06 15:29:00
1,TAM,SBRJ,SBGR,2022-01-07 14:20:00,2022-01-07 14:47:00,2022-01-07 15:25:00,2022-01-07 15:42:00
2,TAM,SBRJ,SBGR,2022-01-08 14:20:00,2022-01-08 14:12:00,2022-01-08 15:25:00,2022-01-08 15:12:00
3,TAM,SBRJ,SBGR,2022-01-09 14:20:00,2022-01-09 14:16:00,2022-01-09 15:25:00,2022-01-09 15:15:00
4,TAM,SBRJ,SBGR,2022-01-11 14:20:00,2022-01-11 14:20:00,2022-01-11 15:25:00,2022-01-11 15:15:00
